# Análisis Exploratorio — Base de Reclamos
**Scotiabank Peru — Prevención de Fraude**  
Base 8850 / Master File — Toda la data es fraude confirmado

---
**Secciones:**
1. Carga y resumen global
2. Evolución temporal de reclamos
3. Distribución de montos
4. Canal, MCC y tipo de comercio
5. Tiempo hasta el reclamo (DIAS_HASTA_RECLAMO)
6. Correlaciones entre variables numéricas
7. Boxplots: variable numérica × variable categórica
8. BIN extendido y señales de clonación
9. Período de ataque Google (ene-feb 2026)
10. Señales de autofraud (SCORE_AUTOFRAUD)

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

# ── Rutas ────────────────────────────────────────────────────────────────────
REPO   = Path('..').resolve()
PARQUET = REPO / 'data' / 'reclamos_features.parquet'

sys.path.insert(0, str(REPO / 'scripts'))
from config import COLS, SEGMENTO_FOCO, UMBRAL_RECLAMO_TARDIO_DIAS
C = COLS

df = pd.read_parquet(PARQUET)
print(f'Shape         : {df.shape}')
print(f'Segmento foco : {SEGMENTO_FOCO}')
print(f'Columnas      : {df.columns.tolist()}')

---
## 1. Resumen global

In [ ]:
col_monto  = C['monto']
col_cli    = C['id_cliente']
col_com    = C['comercio_nom']
col_bin    = C.get('bin', '')
col_fh     = C['fecha_hora']

df[col_fh] = pd.to_datetime(df[col_fh], errors='coerce')
df[col_monto] = pd.to_numeric(df[col_monto], errors='coerce')

resumen = {
    'Total reclamos'        : len(df),
    'Clientes únicos'       : df[col_cli].nunique(),
    'Comercios únicos'      : df[col_com].nunique(),
    'BINs únicos'           : df[col_bin].nunique() if col_bin in df.columns else 'N/A',
    'Monto total (S/)'      : f"{df[col_monto].sum():,.2f}",
    'Monto promedio (S/)'   : f"{df[col_monto].mean():,.2f}",
    'Monto mediana (S/)'    : f"{df[col_monto].median():,.2f}",
    'Monto P10 (S/)'        : f"{df[col_monto].quantile(0.10):,.2f}",
    'Monto P90 (S/)'        : f"{df[col_monto].quantile(0.90):,.2f}",
    'Período inicio'        : str(df[col_fh].min().date()),
    'Período fin'           : str(df[col_fh].max().date()),
}

for k, v in resumen.items():
    print(f'  {k:<30}: {v}')

if 'DIAS_HASTA_RECLAMO' in df.columns:
    dias = df['DIAS_HASTA_RECLAMO'].dropna()
    print(f'\n  DIAS_HASTA_RECLAMO')
    print(f'    mediana : {dias.median():.0f} días')
    print(f'    P90     : {dias.quantile(0.9):.0f} días')
    print(f'    máximo  : {dias.max():.0f} días')
    if 'FLAG_RECLAMO_TARDIO' in df.columns:
        print(f'    Tardíos (>{UMBRAL_RECLAMO_TARDIO_DIAS}d): {df["FLAG_RECLAMO_TARDIO"].sum():,} ({df["FLAG_RECLAMO_TARDIO"].mean()*100:.1f}%)')

---
## 2. Evolución temporal de reclamos

In [ ]:
# ── Reclamos por mes: volumen y monto ────────────────────────────────────────
df['MES_ANIO_DT'] = df[col_fh].dt.to_period('M').dt.to_timestamp()

mes_agg = (
    df.groupby('MES_ANIO_DT').agg(
        N_RECLAMOS   = (col_monto, 'count'),
        MONTO_TOTAL  = (col_monto, 'sum'),
        MONTO_PROM   = (col_monto, 'mean'),
    ).reset_index()
)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Número de reclamos por mes', 'Monto total reclamado por mes (S/)'],
    shared_xaxes=True,
    vertical_spacing=0.12,
)

fig.add_trace(go.Bar(
    x=mes_agg['MES_ANIO_DT'], y=mes_agg['N_RECLAMOS'],
    name='N° reclamos', marker_color='steelblue',
    text=mes_agg['N_RECLAMOS'], textposition='outside',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=mes_agg['MES_ANIO_DT'], y=mes_agg['MONTO_TOTAL'].round(0),
    name='Monto total', marker_color='salmon',
    text=mes_agg['MONTO_TOTAL'].map(lambda x: f'S/{x:,.0f}'), textposition='outside',
), row=2, col=1)

# Sombrear período Google
for row in [1, 2]:
    fig.add_vrect(
        x0='2026-01-01', x1='2026-02-28',
        fillcolor='orange', opacity=0.12, line_width=0,
        annotation_text='Ataque Google', annotation_position='top left',
        row=row, col=1,
    )

fig.update_layout(height=600, showlegend=False, title_text='Evolución temporal de reclamos')
fig.show()

In [ ]:
# ── Reclamos por día de semana y franja horaria ───────────────────────────────
_orden_dias   = ['MON','TUE','WED','THU','FRI','SAT','SUN']
_orden_franjas = ['MADRUGADA','MANANA','TARDE','NOCHE']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if 'DIA_SEMANA_NOM' in df.columns:
    cnt_dia = df['DIA_SEMANA_NOM'].value_counts().reindex(_orden_dias, fill_value=0)
    sns.barplot(x=cnt_dia.index, y=cnt_dia.values, ax=axes[0], color='steelblue')
    axes[0].set_title('Reclamos por día de semana')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('N° reclamos')

if 'FRANJA_HORARIA' in df.columns:
    cnt_franja = df['FRANJA_HORARIA'].value_counts().reindex(_orden_franjas, fill_value=0)
    sns.barplot(x=cnt_franja.index, y=cnt_franja.values, ax=axes[1],
                palette=['#2c3e50','#f39c12','#e74c3c','#8e44ad'])
    axes[1].set_title('Reclamos por franja horaria')
    axes[1].set_xlabel('')
    axes[1].set_ylabel('N° reclamos')

plt.tight_layout()
plt.show()

---
## 3. Distribución de montos

In [ ]:
# ── Histograma interactivo de montos ─────────────────────────────────────────
_monto_cap = df[col_monto].clip(upper=df[col_monto].quantile(0.99))  # capear outliers extremos para visualización

fig = px.histogram(
    df.assign(monto_cap=_monto_cap),
    x='monto_cap',
    nbins=60,
    title='Distribución de montos reclamados (hasta P99)',
    labels={'monto_cap': 'Monto (S/)'},
    color_discrete_sequence=['steelblue'],
)
fig.update_layout(bargap=0.05)
fig.show()

In [ ]:
# ── Distribución de montos por RANGO_MONTO_CAT ────────────────────────────────
if 'RANGO_MONTO_CAT' in df.columns:
    orden_rango = ['BAJO','MEDIO','ALTO','MUY_ALTO']
    cnt_rango   = df['RANGO_MONTO_CAT'].astype(str).value_counts().reindex(orden_rango).fillna(0)

    fig = px.pie(
        values=cnt_rango.values,
        names=cnt_rango.index,
        title='Proporción de reclamos por rango de monto',
        color_discrete_sequence=px.colors.sequential.Blues_r,
    )
    fig.update_traces(textinfo='percent+label')
    fig.show()

In [ ]:
# ── Deciles de monto ─────────────────────────────────────────────────────────
deciles = df[col_monto].quantile([i/10 for i in range(1, 11)])
deciles.index = [f'P{int(i*100)}' for i in deciles.index]

fig = px.bar(
    x=deciles.index, y=deciles.values,
    title='Percentiles de monto reclamado',
    labels={'x': 'Percentil', 'y': 'Monto (S/)'},
    color=deciles.values,
    color_continuous_scale='Blues',
    text=deciles.values.round(2),
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

---
## 4. Canal, MCC y comercios

In [ ]:
# ── Reclamos por canal ────────────────────────────────────────────────────────
if 'CANAL_TEXTO' in df.columns:
    cnt_canal = df.groupby('CANAL_TEXTO')[col_monto].agg(['count','sum','mean']).reset_index()
    cnt_canal.columns = ['Canal','N_Reclamos','Monto_Total','Monto_Prom']
    cnt_canal = cnt_canal.sort_values('N_Reclamos', ascending=False)

    fig = make_subplots(rows=1, cols=2, subplot_titles=['N° reclamos por canal', 'Monto total por canal (S/)'])
    fig.add_trace(go.Bar(x=cnt_canal['Canal'], y=cnt_canal['N_Reclamos'],
                         marker_color='steelblue', text=cnt_canal['N_Reclamos'], textposition='outside',
                         name='N° reclamos'), row=1, col=1)
    fig.add_trace(go.Bar(x=cnt_canal['Canal'], y=cnt_canal['Monto_Total'].round(0),
                         marker_color='salmon', name='Monto total'), row=1, col=2)
    fig.update_layout(height=420, showlegend=False, title_text='Distribución por canal')
    fig.show()

In [ ]:
# ── Top MCC categorías ────────────────────────────────────────────────────────
if 'MCC_CATEGORIA' in df.columns:
    cnt_mcc = (
        df.groupby('MCC_CATEGORIA')[col_monto]
        .agg(N_Reclamos='count', Monto_Total='sum')
        .reset_index()
        .sort_values('N_Reclamos', ascending=True)
        .tail(15)
    )
    fig = px.bar(
        cnt_mcc, x='N_Reclamos', y='MCC_CATEGORIA', orientation='h',
        title='Top 15 categorías MCC por número de reclamos',
        color='N_Reclamos', color_continuous_scale='Blues',
        labels={'MCC_CATEGORIA': 'Categoría MCC', 'N_Reclamos': 'N° reclamos'},
        text='N_Reclamos',
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(height=500, coloraxis_showscale=False)
    fig.show()

In [ ]:
# ── Top 20 comercios por N° de reclamos ──────────────────────────────────────
top_com = (
    df.groupby(col_com)[col_monto]
    .agg(N_Reclamos='count', Monto_Total='sum', Monto_Prom='mean')
    .reset_index()
    .sort_values('N_Reclamos', ascending=True)
    .tail(20)
)

fig = px.bar(
    top_com, x='N_Reclamos', y=col_com, orientation='h',
    title='Top 20 comercios por número de reclamos',
    color='Monto_Prom',
    color_continuous_scale='Reds',
    labels={col_com: 'Comercio', 'N_Reclamos': 'N° reclamos', 'Monto_Prom': 'Monto prom (S/)'},
    hover_data=['Monto_Total', 'Monto_Prom'],
    text='N_Reclamos',
)
fig.update_traces(textposition='outside')
fig.update_layout(height=600)
fig.show()

---
## 5. Tiempo hasta el reclamo (DIAS_HASTA_RECLAMO)

In [ ]:
if 'DIAS_HASTA_RECLAMO' in df.columns:
    dias_validos = df['DIAS_HASTA_RECLAMO'].dropna()

    # Histograma interactivo
    fig = px.histogram(
        x=dias_validos,
        nbins=90,
        title='Distribución: días entre la transacción y el reclamo',
        labels={'x': 'Días hasta el reclamo'},
        color_discrete_sequence=['steelblue'],
    )
    # Líneas de referencia
    for dias_ref, label, color in [
        (7,  'Reclamo rápido (7d)',  'green'),
        (30, '30 días',             'orange'),
        (60, f'Umbral tardío ({60}d)', 'red'),
    ]:
        fig.add_vline(x=dias_ref, line_dash='dash', line_color=color,
                      annotation_text=label, annotation_position='top right')

    fig.update_layout(bargap=0.02)
    fig.show()

In [ ]:
if 'BUCKET_RECLAMO' in df.columns:
    orden_bucket = ['INMEDIATO','RAPIDO','NORMAL','TARDIO','MUY_TARDIO','SIN_DATO']
    cnt_bucket   = df['BUCKET_RECLAMO'].value_counts().reindex(orden_bucket).fillna(0)

    colores_bucket = ['#2ecc71','#27ae60','#3498db','#e67e22','#e74c3c','#bdc3c7']

    fig = px.pie(
        values=cnt_bucket.values,
        names=cnt_bucket.index,
        title='Distribución por BUCKET_RECLAMO',
        color_discrete_sequence=colores_bucket,
    )
    fig.update_traces(textinfo='percent+label+value')
    fig.show()

    print('\nInterpretación:')
    print('  INMEDIATO (≤3d)   → detectado al instante, fraude evidente para el cliente')
    print('  RAPIDO    (4-7d)  → vio el cargo en la app o notificación')
    print('  NORMAL    (8-60d) → revisó su estado de cuenta mensual')
    print('  TARDIO    (>60d)  → ⚠️  candidato a autofraud / friendly fraud')
    print('  MUY_TARDIO(>90d)  → 🚨  señal fuerte de autofraud')

---
## 6. Correlaciones entre variables numéricas

In [ ]:
# ── Variables numéricas candidatas para el heatmap ────────────────────────────
VARS_NUMERICAS = [
    col_monto,
    'HORA_DIA', 'DIA_SEMANA',
    'DIAS_HASTA_RECLAMO', 'SEMANAS_HASTA_RECLAMO',
    'GAP_MINUTOS', 'GAP_DIAS',
    'N_RECLAMOS_CLIENTE', 'MONTO_TOTAL_RECLAMOS', 'N_COMERCIOS_RECLAMO',
    'N_RECLAMOS_BIN_DIA', 'N_RECLAMOS_BIN_PERIODO',
    'N_TARJETAS_MISMO_BIN12_DIA',
    'ZSCORE_MONTO', 'ZSCORE_MONTO_CLIENTE',
    'SCORE_AUTOFRAUD',
]

vars_disponibles = [v for v in VARS_NUMERICAS if v in df.columns]

df_num = df[vars_disponibles].apply(pd.to_numeric, errors='coerce')

# Eliminar columnas con >70% nulos
df_num = df_num.loc[:, df_num.isna().mean() < 0.70]

corr = df_num.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))  # solo mitad inferior

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 8},
)
ax.set_title('Matriz de correlación — variables numéricas', fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

# Top correlaciones con monto
if col_monto in corr:
    top_corr = corr[col_monto].drop(col_monto).abs().sort_values(ascending=False).head(10)
    print(f'\nTop correlaciones con {col_monto}:')
    for v, r in top_corr.items():
        print(f'  {v:<45}: {r:.3f}')

In [ ]:
# ── Pairplot: variables clave para clustering ──────────────────────────────────
VARS_PAIR = [
    col_monto,
    'DIAS_HASTA_RECLAMO',
    'GAP_MINUTOS',
    'N_RECLAMOS_CLIENTE',
    'HORA_DIA',
    'SCORE_AUTOFRAUD',
]

vars_pair_ok = [v for v in VARS_PAIR if v in df.columns]

hue_col = 'BUCKET_RECLAMO' if 'BUCKET_RECLAMO' in df.columns else None

df_pair = df[vars_pair_ok + ([hue_col] if hue_col else [])].dropna().sample(
    n=min(3000, len(df)), random_state=42
)

# Renombrar para que se lean bien en los ejes
rename_pair = {
    col_monto: 'MONTO',
    'DIAS_HASTA_RECLAMO': 'DIAS_RECLAMO',
    'GAP_MINUTOS': 'GAP_MIN',
    'N_RECLAMOS_CLIENTE': 'N_REC_CLI',
    'HORA_DIA': 'HORA',
}
df_pair = df_pair.rename(columns=rename_pair)
hue_col_r = rename_pair.get(hue_col, hue_col)

g = sns.pairplot(
    df_pair,
    hue=hue_col if hue_col not in rename_pair else None,  # BUCKET_RECLAMO no se renombra
    hue_order=['INMEDIATO','RAPIDO','NORMAL','TARDIO','MUY_TARDIO'] if hue_col else None,
    plot_kws={'alpha': 0.4, 's': 20},
    diag_kind='kde',
    corner=True,
)
g.figure.suptitle('Pairplot — variables clave (muestra 3k, color = BUCKET_RECLAMO)', y=1.01)
plt.show()

---
## 7. Boxplots: variable numérica × variable categórica

In [ ]:
# ── Monto por CANAL y por FRANJA HORARIA ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

_monto_cap = df[col_monto].clip(upper=df[col_monto].quantile(0.97))
df_box = df.copy()
df_box['_monto_cap'] = _monto_cap

if 'CANAL_TEXTO' in df.columns:
    _orden_canal = df_box.groupby('CANAL_TEXTO')['_monto_cap'].median().sort_values(ascending=False).index
    sns.boxplot(
        data=df_box, x='CANAL_TEXTO', y='_monto_cap',
        order=_orden_canal, ax=axes[0], palette='Blues_d',
        flierprops=dict(marker='o', markersize=3, alpha=0.3),
    )
    axes[0].set_title('Monto reclamado por canal')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Monto (S/)')
    axes[0].tick_params(axis='x', rotation=30)

if 'FRANJA_HORARIA' in df.columns:
    _orden_franja = ['MADRUGADA','MANANA','TARDE','NOCHE']
    sns.boxplot(
        data=df_box, x='FRANJA_HORARIA', y='_monto_cap',
        order=_orden_franja, ax=axes[1],
        palette=['#2c3e50','#f39c12','#e74c3c','#8e44ad'],
        flierprops=dict(marker='o', markersize=3, alpha=0.3),
    )
    axes[1].set_title('Monto reclamado por franja horaria')
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Monto (S/)')

plt.suptitle('Distribución de montos por categoría (capado en P97)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Monto y DIAS_HASTA_RECLAMO por BUCKET_RECLAMO ────────────────────────────
if 'BUCKET_RECLAMO' in df.columns:
    orden_bucket = ['INMEDIATO','RAPIDO','NORMAL','TARDIO','MUY_TARDIO']
    _df_b = df_box[df_box['BUCKET_RECLAMO'].isin(orden_bucket)]

    fig = make_subplots(rows=1, cols=2,
                         subplot_titles=['Monto por BUCKET_RECLAMO',
                                         'DIAS_HASTA_RECLAMO por bucket'])

    for bucket in orden_bucket:
        _sub = _df_b[_df_b['BUCKET_RECLAMO'] == bucket]
        fig.add_trace(go.Box(
            y=_sub['_monto_cap'], name=bucket, boxmean=True,
        ), row=1, col=1)

        if 'DIAS_HASTA_RECLAMO' in _sub.columns:
            fig.add_trace(go.Box(
                y=_sub['DIAS_HASTA_RECLAMO'].dropna(), name=bucket, showlegend=False,
            ), row=1, col=2)

    fig.update_layout(height=450, title_text='Análisis por BUCKET_RECLAMO',
                      showlegend=True)
    fig.show()

In [ ]:
# ── Monto por tipo de entrada (TIPO_ENTRADA) ──────────────────────────────────
if 'TIPO_ENTRADA' in df.columns:
    fig = px.box(
        df_box[df_box['TIPO_ENTRADA'] != 'Sin dato'],
        x='TIPO_ENTRADA', y='_monto_cap',
        color='TIPO_ENTRADA',
        title='Monto reclamado por tipo de entrada (entry mode)',
        labels={'_monto_cap': 'Monto (S/)', 'TIPO_ENTRADA': 'Tipo de entrada'},
        points=False,
    )
    fig.update_layout(xaxis_tickangle=-30, showlegend=False)
    fig.show()

---
## 8. BIN extendido y señales de clonación

In [ ]:
if col_bin and col_bin in df.columns:
    # Top 20 BINs por número de reclamos
    top_bin = (
        df.groupby(col_bin)[col_monto]
        .agg(N_Reclamos='count', Monto_Total='sum', Monto_Prom='mean')
        .reset_index()
        .sort_values('N_Reclamos', ascending=True)
        .tail(20)
    )

    fig = px.bar(
        top_bin, x='N_Reclamos', y=col_bin, orientation='h',
        title='Top 20 BINs por número de reclamos',
        color='Monto_Prom',
        color_continuous_scale='Reds',
        labels={col_bin: 'BIN', 'N_Reclamos': 'N° reclamos', 'Monto_Prom': 'Monto prom (S/)'},
        hover_data=['Monto_Total', 'Monto_Prom'],
        text='N_Reclamos',
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(height=560)
    fig.show()

In [ ]:
# ── Análisis FLAG_BIN12_REPETIDO_DIA ─────────────────────────────────────────
if 'FLAG_BIN12_REPETIDO_DIA' in df.columns:
    cnt_bin12 = df['FLAG_BIN12_REPETIDO_DIA'].value_counts()
    pct_bin12 = df['FLAG_BIN12_REPETIDO_DIA'].mean() * 100

    print(f'FLAG_BIN12_REPETIDO_DIA (≥2 tarjetas del mismo BIN12 en el día):')
    print(f'  Activado : {cnt_bin12.get(1, 0):,} reclamos ({pct_bin12:.1f}%)')
    print(f'  No activo: {cnt_bin12.get(0, 0):,} reclamos')
    print()

    # Distribución de N_TARJETAS_MISMO_BIN12_DIA
    if 'N_TARJETAS_MISMO_BIN12_DIA' in df.columns:
        dist_n = df['N_TARJETAS_MISMO_BIN12_DIA'].value_counts().sort_index().head(20)

        fig = px.bar(
            x=dist_n.index.astype(str), y=dist_n.values,
            title='Distribución: N° de tarjetas distintas con el mismo BIN12 en el día',
            labels={'x': 'N° tarjetas con mismo BIN12', 'y': 'N° reclamos'},
            color=dist_n.values,
            color_continuous_scale='Reds',
            text=dist_n.values,
        )
        fig.update_traces(textposition='outside')
        fig.update_layout(coloraxis_showscale=False)
        fig.show()

---
## 9. Período de ataque Google (ene-feb 2026)

In [ ]:
if 'ES_PERIODO_GOOGLE' in df.columns:
    df_g  = df[df['ES_PERIODO_GOOGLE'] == 1]
    df_ng = df[df['ES_PERIODO_GOOGLE'] == 0]

    print('COMPARATIVA: período Google vs resto del período')
    print(f'  {"Métrica":<35} {"Google (ene-feb)":>18} {"Resto":>18}')
    print('  ' + '-'*72)
    metricas = [
        ('N° reclamos',      len(df_g),                 len(df_ng)),
        ('Monto total (S/)', df_g[col_monto].sum(),     df_ng[col_monto].sum()),
        ('Monto promedio',   df_g[col_monto].mean(),    df_ng[col_monto].mean()),
        ('Monto mediana',    df_g[col_monto].median(),  df_ng[col_monto].median()),
    ]
    for label, val_g, val_ng in metricas:
        if isinstance(val_g, float):
            print(f'  {label:<35} {val_g:>18,.2f} {val_ng:>18,.2f}')
        else:
            print(f'  {label:<35} {val_g:>18,} {val_ng:>18,}')

    # Boxplot: monto Google vs resto
    df_comp = df[[col_monto, 'ES_PERIODO_GOOGLE']].copy()
    df_comp['Período'] = df_comp['ES_PERIODO_GOOGLE'].map({1: 'Google (ene-feb 2026)', 0: 'Resto del período'})
    df_comp['_monto_cap'] = df_comp[col_monto].clip(upper=df_comp[col_monto].quantile(0.97))

    fig = px.box(
        df_comp, x='Período', y='_monto_cap',
        color='Período',
        color_discrete_map={'Google (ene-feb 2026)': 'orange', 'Resto del período': 'steelblue'},
        title='Distribución de monto: período Google vs resto',
        labels={'_monto_cap': 'Monto (S/)'},
        points='outliers',
    )
    fig.update_layout(showlegend=False)
    fig.show()

In [ ]:
# ── Perfil horario: Google vs resto ──────────────────────────────────────────
if 'ES_PERIODO_GOOGLE' in df.columns and 'HORA_DIA' in df.columns:
    hora_g  = df[df['ES_PERIODO_GOOGLE'] == 1]['HORA_DIA'].value_counts().sort_index()
    hora_ng = df[df['ES_PERIODO_GOOGLE'] == 0]['HORA_DIA'].value_counts().sort_index()

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hora_g.index,  y=hora_g.values,  mode='lines+markers',
                             name='Google (ene-feb)', line=dict(color='orange')))
    fig.add_trace(go.Scatter(x=hora_ng.index, y=hora_ng.values, mode='lines+markers',
                             name='Resto del período', line=dict(color='steelblue')))
    fig.update_layout(
        title='Perfil horario: período Google vs resto',
        xaxis_title='Hora del día',
        yaxis_title='N° reclamos',
        xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    )
    fig.show()

---
## 10. Señales de autofraud

In [ ]:
if 'SCORE_AUTOFRAUD' in df.columns:
    cnt_score = df['SCORE_AUTOFRAUD'].value_counts().sort_index()

    fig = px.bar(
        x=cnt_score.index.astype(str), y=cnt_score.values,
        title='Distribución de SCORE_AUTOFRAUD (0=fraude claro, 5=máxima señal de autofraud)',
        labels={'x': 'Score Autofraud', 'y': 'N° reclamos'},
        color=cnt_score.index,
        color_continuous_scale='RdYlGn_r',
        text=cnt_score.values,
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(coloraxis_showscale=False)
    fig.show()

    if 'FLAG_CANDIDATO_AUTOFRAUD' in df.columns:
        n_autofraud = df['FLAG_CANDIDATO_AUTOFRAUD'].sum()
        pct = n_autofraud / len(df) * 100
        print(f'\nFLAG_CANDIDATO_AUTOFRAUD (score ≥ 4):')
        print(f'  {n_autofraud:,} reclamos ({pct:.1f}%) → candidatos a revisión por autofraud')

        # Comparativa candidatos vs no candidatos
        comp = df.groupby('FLAG_CANDIDATO_AUTOFRAUD')[col_monto].agg(['count','mean','median']).round(2)
        comp.index = comp.index.map({0: 'Fraude regular', 1: 'Candidato autofraud'})
        print()
        print(comp.to_string())

In [ ]:
# ── Flags individuales de autofraud ──────────────────────────────────────────
flags_autofraud = [
    ('FLAG_RECLAMO_TARDIO',       f'Reclamo tardío (>{UMBRAL_RECLAMO_TARDIO_DIAS}d)'),
    ('FLAG_MONTO_FUERA_PATRON',   'Monto fuera del patrón del cliente'),
    ('ES_TARJETA_PRESENTE',       'Tarjeta presente (fraude físico)'),
    ('FLAG_BIN12_REPETIDO_DIA',   'BIN12 repetido ese día (clonación)'),
    ('FLAG_MONTO_BAJO',           'Monto bajo (card testing)'),
    ('FLAG_CLIENTE_MULTI_RECLAMO','Cliente con múltiples reclamos'),
    ('ES_INTERNACIONAL',          'Transacción internacional'),
    ('ES_TOKENIZADA',             'Billetera digital (tokenizada)'),
]

labels_flags  = []
valores_flags = []
for col_f, label_f in flags_autofraud:
    if col_f in df.columns:
        labels_flags.append(label_f)
        valores_flags.append(df[col_f].sum())

if labels_flags:
    orden_flags = sorted(zip(valores_flags, labels_flags), reverse=False)
    vals_ord, labs_ord = zip(*orden_flags)

    fig = px.bar(
        x=list(vals_ord), y=list(labs_ord),
        orientation='h',
        title='Activaciones de señales de fraude y autofraud',
        labels={'x': 'N° reclamos con señal activa', 'y': ''},
        color=list(vals_ord),
        color_continuous_scale='RdBu_r',
        text=[f'{v:,}' for v in vals_ord],
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(height=420, coloraxis_showscale=False)
    fig.show()

---
## 11. Export a Excel — resumen tabular para gerente

In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

OUTPUT_EXCEL = REPO / 'output' / 'eda_reclamos.xlsx'
OUTPUT_EXCEL.parent.mkdir(exist_ok=True)

# Estilo Scotiabank (rojo)
COLOR_TITULO  = 'C8102E'
COLOR_HEADER  = '2C2C2C'
COLOR_ALTROW  = 'FFF0F0'

def estilo_wb(wb, hoja_nombre, df_tab, titulo):
    ws = wb.create_sheet(title=hoja_nombre)

    # Título
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(df_tab.columns))
    ws.cell(1, 1, titulo).font       = Font(bold=True, size=13, color='FFFFFF')
    ws.cell(1, 1).fill               = PatternFill('solid', fgColor=COLOR_TITULO)
    ws.cell(1, 1).alignment          = Alignment(horizontal='center')

    # Headers
    for j, col in enumerate(df_tab.columns, 1):
        c = ws.cell(2, j, col)
        c.font      = Font(bold=True, color='FFFFFF')
        c.fill      = PatternFill('solid', fgColor=COLOR_HEADER)
        c.alignment = Alignment(horizontal='center')

    # Data
    for i, row in enumerate(df_tab.itertuples(index=False), 3):
        fill = PatternFill('solid', fgColor=COLOR_ALTROW) if i % 2 == 0 else None
        for j, val in enumerate(row, 1):
            c = ws.cell(i, j, val)
            if fill:
                c.fill = fill
            c.alignment = Alignment(horizontal='right' if isinstance(val, (int, float)) else 'left')

    # Autofit
    for j in range(1, len(df_tab.columns) + 1):
        max_len = max(
            [len(str(ws.cell(r, j).value or '')) for r in range(1, ws.max_row + 1)]
        )
        ws.column_dimensions[get_column_letter(j)].width = min(max_len + 3, 50)

    return ws


wb = openpyxl.Workbook()
wb.remove(wb.active)  # borrar hoja vacía

# ── Hoja 1: Resumen global ────────────────────────────────────────────────────
df_res = pd.DataFrame(list(resumen.items()), columns=['Métrica', 'Valor'])
estilo_wb(wb, '01_Resumen', df_res, f'Resumen Global — Reclamos {SEGMENTO_FOCO}')

# ── Hoja 2: Por mes ───────────────────────────────────────────────────────────
df_mes_tab = mes_agg.copy()
df_mes_tab['MES_ANIO_DT'] = df_mes_tab['MES_ANIO_DT'].dt.strftime('%Y-%m')
df_mes_tab.columns = ['Mes', 'N_Reclamos', 'Monto_Total_S/', 'Monto_Prom_S/']
df_mes_tab = df_mes_tab.round(2)
estilo_wb(wb, '02_Por_Mes', df_mes_tab, 'Reclamos por Mes')

# ── Hoja 3: Por canal ─────────────────────────────────────────────────────────
if 'CANAL_TEXTO' in df.columns:
    df_c = df.groupby('CANAL_TEXTO')[col_monto].agg(
        N_Reclamos='count', Monto_Total='sum', Monto_Prom='mean'
    ).reset_index().sort_values('N_Reclamos', ascending=False).round(2)
    estilo_wb(wb, '03_Por_Canal', df_c, 'Reclamos por Canal')

# ── Hoja 4: Por MCC ───────────────────────────────────────────────────────────
if 'MCC_CATEGORIA' in df.columns:
    df_mcc_tab = df.groupby('MCC_CATEGORIA')[col_monto].agg(
        N_Reclamos='count', Monto_Total='sum', Monto_Prom='mean'
    ).reset_index().sort_values('N_Reclamos', ascending=False).round(2)
    estilo_wb(wb, '04_Por_MCC', df_mcc_tab, 'Reclamos por Categoría MCC')

# ── Hoja 5: Top comercios ─────────────────────────────────────────────────────
df_top_com_tab = df.groupby(col_com)[col_monto].agg(
    N_Reclamos='count', Monto_Total='sum', Monto_Prom='mean'
).reset_index().sort_values('N_Reclamos', ascending=False).head(30).round(2)
df_top_com_tab.columns = ['Comercio', 'N_Reclamos', 'Monto_Total_S/', 'Monto_Prom_S/']
estilo_wb(wb, '05_Top_Comercios', df_top_com_tab, 'Top 30 Comercios')

# ── Hoja 6: BUCKET_RECLAMO ────────────────────────────────────────────────────
if 'BUCKET_RECLAMO' in df.columns:
    df_buck_tab = df.groupby('BUCKET_RECLAMO').agg(
        N_Reclamos   = (col_monto, 'count'),
        Pct_Total    = (col_monto, lambda x: f'{len(x)/len(df)*100:.1f}%'),
        Monto_Total  = (col_monto, 'sum'),
        Monto_Prom   = (col_monto, 'mean'),
        Dias_Mediana = ('DIAS_HASTA_RECLAMO', 'median'),
    ).reset_index().round(2)
    estilo_wb(wb, '06_Bucket_Reclamo', df_buck_tab, 'Distribución por Tiempo de Reclamo')

# ── Hoja 7: Top BINs ──────────────────────────────────────────────────────────
if col_bin and col_bin in df.columns:
    df_bin_tab = df.groupby(col_bin)[col_monto].agg(
        N_Reclamos='count', Monto_Total='sum', Monto_Prom='mean'
    ).reset_index().sort_values('N_Reclamos', ascending=False).head(30).round(2)
    df_bin_tab.columns = ['BIN', 'N_Reclamos', 'Monto_Total_S/', 'Monto_Prom_S/']
    estilo_wb(wb, '07_Top_BINs', df_bin_tab, 'Top 30 BINs')

# ── Hoja 8: Señales de autofraud ──────────────────────────────────────────────
if 'SCORE_AUTOFRAUD' in df.columns:
    df_score_tab = df.groupby('SCORE_AUTOFRAUD')[col_monto].agg(
        N_Reclamos='count', Monto_Total='sum', Monto_Prom='mean'
    ).reset_index().round(2)
    df_score_tab['Pct_Total'] = (df_score_tab['N_Reclamos'] / len(df) * 100).round(1).astype(str) + '%'
    estilo_wb(wb, '08_Score_Autofraud', df_score_tab, 'Distribución SCORE_AUTOFRAUD')

wb.save(OUTPUT_EXCEL)
print(f'\n✅ Excel guardado: {OUTPUT_EXCEL}')
print(f'   Hojas: {[s.title for s in wb.sheetnames if hasattr(wb, "sheetnames") else []]}')
